# Large Mixed-Size Reproduction Notebook for Colab

This notebook is the **plan-aligned long run** rather than the quick proof-of-concept.
It is designed to reproduce the later mixed-size experiments described in `doc/plan.md`:

- **Large SFT** on a mixed dataset spanning `3x3` to `7x7`
- **Longer RL** on the harder subset, primarily `5x5` to `7x7`
- evaluation by maze size so we can see where RL helps and where it regresses

The main lesson from the plan was not just that RL helps. It was:

- **SFT is the main lever** for teaching maze-conditioned behavior
- **RL is useful as a targeted refinement stage** once the SFT baseline is strong enough

## Target Configuration from the Plan

This notebook aims to match the strongest large mixed-size recipe that was documented:

- **SFT training set**: `7,500` mazes
  - `3x3`: `2,000`
  - `4x4`: `2,000`
  - `5x5`: `2,000`
  - `6x6`: `1,000`
  - `7x7`: `500`
- **SFT training**: `5000` iterations, `batch_size=8`, `lr=5e-5`
- **RL target sizes**: `5x5` to `7x7`
- **RL training**: `1000` steps, `16` generations, `temperature=1.2`, `max_tokens=60`

If you are on a tighter Colab budget, there is a switch below to disable `7x7` while keeping
`3x3` through `6x6` mixed training.

In [ ]:
# Colab / notebook setup
!pip -q install transformers accelerate peft datasets sentencepiece matplotlib "pandas==2.2.2"

In [ ]:
import os
import subprocess
import sys
from dataclasses import asdict, dataclass
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch

DEFAULT_REPO_URL = "https://github.com/StephenJHardy/ascii-maze-rl.git"
DEFAULT_REPO_BRANCH = "pytorch-version"
DEFAULT_REPO_DIR = Path("/content/ascii-maze-rl")


def ensure_repo_root() -> Path:
    cwd = Path.cwd()
    if (cwd / "src").exists():
        return cwd

    env_repo = os.environ.get("ASCII_MAZE_RL_REPO")
    if env_repo and (Path(env_repo) / "src").exists():
        return Path(env_repo)

    if DEFAULT_REPO_DIR.exists() and (DEFAULT_REPO_DIR / "src").exists():
        return DEFAULT_REPO_DIR

    print(f"Cloning repo branch {DEFAULT_REPO_BRANCH!r} into {DEFAULT_REPO_DIR} ...")
    subprocess.run(
        ["git", "clone", "--branch", DEFAULT_REPO_BRANCH, DEFAULT_REPO_URL, str(DEFAULT_REPO_DIR)],
        check=True,
    )
    return DEFAULT_REPO_DIR


REPO_ROOT = ensure_repo_root()
os.chdir(REPO_ROOT)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print(f"repo_root={REPO_ROOT}")
print(f"device={'cuda' if torch.cuda.is_available() else 'cpu'} torch={torch.__version__}")

In [ ]:
from src.maze_dataset import MazeRecord
from src.maze_gen import generate
from src.reward import compute_reward
from src.training import RLConfig, SFTConfig, evaluate_policy, load_policy, train_rl, train_sft

## Reproduction Config

`include_7x7=True` matches the plan more closely. If your Colab runtime is too slow,
set it to `False` and rerun the notebook to keep the experiment at `3x3` through `6x6`.

In [ ]:
@dataclass
class ReproConfig:
    model_name: str = "Qwen/Qwen2.5-0.5B-Instruct"
    include_7x7: bool = True
    sft_batch_size: int = 8
    sft_iters: int = 5000
    sft_lr: float = 5e-5
    lora_rank: int = 16
    rl_steps: int = 1000
    rl_generations: int = 16
    rl_temperature: float = 1.2
    rl_max_tokens: int = 60
    rl_lr: float = 5e-6
    rl_beta: float = 0.1
    eval_samples: int = 1
    output_root: str = "colab_runs_large_repro"

CFG = ReproConfig()
OUTPUT_ROOT = Path(CFG.output_root)
SFT_DIR = OUTPUT_ROOT / "sft_large"
RL_DIR = OUTPUT_ROOT / "rl_large"
CFG

## Dataset Specs

These specs mirror the plan's large mixed-size SFT run. The evaluation split uses a disjoint
seed range and keeps the `3x3` set at `192` mazes, matching the plan's tables.

In [ ]:
BASE_TRAIN_SPECS = [
    {"width": 3, "height": 3, "count": 2000, "seed_start": 0},
    {"width": 4, "height": 4, "count": 2000, "seed_start": 100_000},
    {"width": 5, "height": 5, "count": 2000, "seed_start": 200_000},
    {"width": 6, "height": 6, "count": 1000, "seed_start": 300_000},
    {"width": 7, "height": 7, "count": 500, "seed_start": 400_000},
]

BASE_EVAL_SPECS = [
    {"width": 3, "height": 3, "count": 192, "seed_start": 1_000_000},
    {"width": 4, "height": 4, "count": 200, "seed_start": 1_100_000},
    {"width": 5, "height": 5, "count": 200, "seed_start": 1_200_000},
    {"width": 6, "height": 6, "count": 200, "seed_start": 1_300_000},
    {"width": 7, "height": 7, "count": 200, "seed_start": 1_400_000},
]


def maybe_trim_7x7(specs, include_7x7: bool):
    if include_7x7:
        return specs
    return [spec for spec in specs if spec["width"] != 7]


TRAIN_SPECS = maybe_trim_7x7(BASE_TRAIN_SPECS, CFG.include_7x7)
EVAL_SPECS = maybe_trim_7x7(BASE_EVAL_SPECS, CFG.include_7x7)
TRAIN_SPECS, EVAL_SPECS[:2]

In [ ]:
def make_records(count: int, seed_start: int, width: int, height: int):
    return [
        MazeRecord.from_maze(generate(width, height, seed=seed_start + i))
        for i in range(count)
    ]


def make_records_from_specs(specs):
    records = []
    for spec in specs:
        records.extend(make_records(**spec))
    return records


train_records = make_records_from_specs(TRAIN_SPECS)
eval_records = make_records_from_specs(EVAL_SPECS)
rl_records = [record for record in train_records if record.width >= 5]

pd.DataFrame([
    {"split": "sft_train", "n": len(train_records)},
    {"split": "rl_train", "n": len(rl_records)},
    {"split": "eval", "n": len(eval_records)},
])

## Reference Results from the Plan

We keep the plan's target numbers in the notebook so that the final tables can be compared
against them directly.

In [ ]:
plan_sft_reference = pd.DataFrame([
    {"size": "3x3", "plan_sft_solve_rate": 1.00},
    {"size": "4x4", "plan_sft_solve_rate": 0.88},
    {"size": "5x5", "plan_sft_solve_rate": 0.32},
    {"size": "6x6", "plan_sft_solve_rate": 0.133},
    {"size": "7x7", "plan_sft_solve_rate": 0.05},
])

plan_rl_reference = pd.DataFrame([
    {"size": "3x3", "plan_rl_solve_rate": 0.98},
    {"size": "4x4", "plan_rl_solve_rate": 0.88},
    {"size": "5x5", "plan_rl_solve_rate": 0.26},
    {"size": "6x6", "plan_rl_solve_rate": 0.20},
    {"size": "7x7", "plan_rl_solve_rate": 0.00},
])

if not CFG.include_7x7:
    plan_sft_reference = plan_sft_reference.query("size != '7x7'")
    plan_rl_reference = plan_rl_reference.query("size != '7x7'")

plan_sft_reference.merge(plan_rl_reference, on="size")

## Evaluation Helpers

The repo's evaluation API returns typed result objects. These notebook helpers convert them into
DataFrames and aggregate by size for easier comparison against the plan tables.

In [ ]:
def evaluate_to_frame(policy, records, title: str):
    results, summary = evaluate_policy(
        policy,
        records,
        max_tokens=CFG.rl_max_tokens,
        temperature=0.0,
        num_samples=CFG.eval_samples,
        verbose=True,
    )
    return pd.DataFrame([asdict(r) for r in results]), pd.Series({"name": title, **asdict(summary)})


def summarize_by_size(df):
    grouped = (
        df.assign(size=lambda x: x["width"].astype(str) + "x" + x["height"].astype(str))
        .groupby("size", as_index=False)
        .agg(
            n=("maze_id", "count"),
            solve_rate=("solved", "mean"),
            parse_rate=("moves_parsed", lambda col: col.notna().mean()),
            mean_reward=("reward", "mean"),
            mean_progress=("progress", "mean"),
        )
        .sort_values("size")
    )
    return grouped

## Baseline Evaluation

This is not the main point of the notebook, but it is useful to have a baseline row before the
large SFT run starts.

In [ ]:
base_policy = load_policy(backend="auto", model=CFG.model_name)
base_eval_df, base_eval_summary = evaluate_to_frame(base_policy, eval_records, title="base_eval")
base_by_size = summarize_by_size(base_eval_df)
base_by_size

## Large Mixed-Size SFT

This is the main stage from the plan. It is intentionally long-running. The target is to push the
mixed-size SFT baseline high enough that RL becomes a refinement step rather than a rescue step.

In [ ]:
sft_output_dir = train_sft(
    SFTConfig(
        model=CFG.model_name,
        records=train_records,
        val_records=eval_records,
        iters=CFG.sft_iters,
        batch_size=CFG.sft_batch_size,
        lr=CFG.sft_lr,
        lora_rank=CFG.lora_rank,
        output_dir=SFT_DIR,
        backend="auto",
    )
)
sft_output_dir

In [ ]:
sft_policy = load_policy(backend="auto", model=CFG.model_name, adapter_path=sft_output_dir, lora_rank=CFG.lora_rank)
sft_eval_df, sft_eval_summary = evaluate_to_frame(sft_policy, eval_records, title="sft_eval")
sft_by_size = summarize_by_size(sft_eval_df)

sft_comparison = sft_by_size.merge(plan_sft_reference, on="size", how="left")
sft_comparison["delta_vs_plan"] = sft_comparison["solve_rate"] - sft_comparison["plan_sft_solve_rate"]
sft_comparison

## Longer RL on the Harder Sizes

This mirrors the later plan run: use the large SFT checkpoint as the starting point, then run a
longer RL phase on the harder subset. The reward function remains exposed here on purpose so a
future notebook can turn this into a reward-design exercise.

In [ ]:
reward_fn = None

rl_output_dir = train_rl(
    RLConfig(
        model=CFG.model_name,
        adapters=sft_output_dir,
        records=rl_records,
        max_steps=CFG.rl_steps,
        num_generations=CFG.rl_generations,
        max_tokens=CFG.rl_max_tokens,
        temperature=CFG.rl_temperature,
        lr=CFG.rl_lr,
        beta=CFG.rl_beta,
        lora_rank=CFG.lora_rank,
        log_interval=10,
        save_interval=1000,
        output_dir=RL_DIR,
        backend="auto",
    ),
    reward=reward_fn,
)
rl_output_dir

In [ ]:
rl_policy = load_policy(backend="auto", model=CFG.model_name, adapter_path=rl_output_dir, lora_rank=CFG.lora_rank)
rl_eval_df, rl_eval_summary = evaluate_to_frame(rl_policy, eval_records, title="rl_eval")
rl_by_size = summarize_by_size(rl_eval_df)

rl_comparison = rl_by_size.merge(plan_rl_reference, on="size", how="left")
rl_comparison["delta_vs_plan"] = rl_comparison["solve_rate"] - rl_comparison["plan_rl_solve_rate"]
rl_comparison

## End-to-End Comparison

This table is the one to keep. It compares the base model, the large mixed-size SFT run, and the
longer RL refinement pass.

In [ ]:
overall = pd.DataFrame([
    base_eval_summary,
    sft_eval_summary,
    rl_eval_summary,
])[["name", "total", "solved", "parseable", "mean_reward", "mean_progress"]]
overall

In [ ]:
merged = (
    sft_by_size[["size", "solve_rate"]]
    .rename(columns={"solve_rate": "sft_solve_rate"})
    .merge(
        rl_by_size[["size", "solve_rate"]].rename(columns={"solve_rate": "rl_solve_rate"}),
        on="size",
        how="outer",
    )
    .merge(plan_sft_reference, on="size", how="left")
    .merge(plan_rl_reference, on="size", how="left")
    .sort_values("size")
)
merged

In [ ]:
rl_log_path = Path(rl_output_dir) / "log.json"
rl_logs = pd.read_json(rl_log_path) if rl_log_path.exists() else pd.DataFrame()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
if not rl_logs.empty:
    axes[0].plot(rl_logs["step"], rl_logs["reward_mean"], label="mean reward")
    axes[0].plot(rl_logs["step"], rl_logs["reward_max"], label="max reward")
    axes[0].legend()
axes[0].set_title("RL training reward curves")
axes[0].set_xlabel("step")

plot_df = merged.copy()
plot_df = plot_df.set_index("size")[["sft_solve_rate", "rl_solve_rate", "plan_sft_solve_rate", "plan_rl_solve_rate"]]
plot_df.plot(kind="bar", ax=axes[1], rot=0)
axes[1].set_title("Solve rate by size")
axes[1].set_ylabel("solve rate")
plt.tight_layout()
plt.show()

## How to Interpret the Outcome

The plan's final mixed-size story was subtle:

- the **large SFT run** was the biggest jump in quality
- the **longer RL run** helped `6x6`, which was the main hard-size win
- that same RL run **hurt `5x5`**, which is exactly why size-specific reporting matters

So a successful reproduction does **not** necessarily mean every size improves after RL. The more
honest success criterion is whether the run reproduces the same pattern:

- SFT is the dominant improvement
- RL helps in the target harder regime
- RL can regress neighboring sizes when trained on a narrow subset

## Variants to Try Next

If this exact reproduction is too slow for your hardware, these are the first controlled downgrades:

- set `include_7x7=False` to keep the run at `3x3` through `6x6`
- reduce `sft_iters` from `5000` to `2000` to reproduce the earlier large SFT run
- reduce `rl_steps` from `1000` to `500` to reproduce the shorter RL run

If the goal is a teaching notebook on reward design, keep this notebook fixed and vary only the
`reward_fn` argument passed into `train_rl(...)`.